<a href="https://colab.research.google.com/github/Janpu-Hou/Green-Learning-Basic/blob/main/KIMI_GL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install imblearn library for handling imbalanced datasets
!pip install imblearn

In [ ]:
#!/usr/bin/env python3
"""
================================================================================
Green Learning (GL) Multi-Class Classifier for IoT-23 (Preprocessed Data)
================================================================================
A complete, end-to-end Green Learning pipeline for IoT network intrusion
detection on the IoT-23 dataset.

This version is modified to ingest fully numeric, pre-engineered data
generated by the `iot23_preprocessor.py` script.

Pipeline:
  Stage 1: Unsupervised Representation Learning    → PCA subspace approximation
  Stage 2: Supervised Discriminant Feature Selection → DFT (Discriminant Feature Test)
  Stage 3: Supervised Decision Learning              → SLM Forest / SLM Boost

Usage:
    python gl_iot23.py --train_path ./iot23_processed/iot23_train.csv --test_path ./iot23_processed/iot23_test.csv

Requirements: numpy, pandas, scikit-learn
================================================================================
"""

import os
import sys
import argparse
import time
import warnings
from collections import Counter
from typing import Tuple, List, Optional

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.feature_selection import SelectorMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, accuracy_score, f1_score)
from sklearn.utils.validation import check_X_y, check_array

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# IoT-23 Unified Classes (Matches the Preprocessor Output)
IOT23_CLASSES = [
    'Benign',
    'Attack',
    'C&C',
    'DDoS',
    'FileDownload',
    'Okiru',
    'PortScan',
    'Torii'
]


# =============================================================================
# SECTION 1: STAGE 1 — UNSUPERVISED REPRESENTATION LEARNING (PCA)
# =============================================================================

class SubspaceApproximation(BaseEstimator, TransformerMixin):
    """
    Stage 1: Unsupervised Subspace Approximation via PCA.
    Reduces spectral dimension while retaining specified variance.
    """
    def __init__(self, variance_threshold: float = 0.95):
        self.variance_threshold = variance_threshold
        self.scaler = StandardScaler()
        self.pca = None
        self.n_components_ = None
        self.explained_variance_ = None

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=np.float64)
        X_scaled = self.scaler.fit_transform(X)
        self.pca = PCA(n_components=self.variance_threshold, random_state=RANDOM_STATE)
        self.pca.fit(X_scaled)
        self.n_components_ = self.pca.n_components_
        self.explained_variance_ = self.pca.explained_variance_ratio_.sum()
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        X_scaled = self.scaler.transform(X)
        return self.pca.transform(X_scaled)

    def fit_transform(self, X, y=None):
        self.fit(X)
        return self.transform(X)


# =============================================================================
# SECTION 2: STAGE 2 — DISCRIMINANT FEATURE TEST (DFT)
# =============================================================================

class DiscriminantFeatureTest(BaseEstimator, SelectorMixin, TransformerMixin):
    """
    Stage 2: Discriminant Feature Test (DFT) for supervised feature selection.
    Evaluates each 1-D feature by partitioning its range and computing the
    minimum weighted entropy across candidate thresholds. Lower DFT loss
    = higher discriminant power.
    """
    def __init__(self, n_bins: int = 16, k: str = 'auto',
                 elbow_factor: float = 0.95, verbose: bool = False):
        self.n_bins = n_bins
        self.k = k
        self.elbow_factor = elbow_factor
        self.verbose = verbose

    def _dft_fast(self, X, y):
        N, P = X.shape
        f_min = X.min(axis=0)
        f_max = X.max(axis=0)
        degenerate = (f_max - f_min) < 1e-10

        b_vals = np.arange(1, self.n_bins).reshape(-1, 1)
        thresholds = f_min + (b_vals / self.n_bins) * (f_max - f_min)

        classes = np.unique(y)
        X_exp = X[np.newaxis, :, :]
        T_exp = thresholds[:, np.newaxis, :]

        left_mask = X_exp < T_exp
        right_mask = ~left_mask
        N_L = left_mask.sum(axis=1)
        N_R = right_mask.sum(axis=1)
        valid = (N_L > 0) & (N_R > 0)

        H = np.zeros((self.n_bins - 1, P))

        for c in classes:
            y_eq_c = (y == c).astype(np.float64)
            p_L = np.tensordot(left_mask.astype(np.float64), y_eq_c, axes=([1], [0]))
            p_L = p_L / (N_L + 1e-10)
            p_L = np.where(valid, p_L, 0)
            p_R = np.tensordot(right_mask.astype(np.float64), y_eq_c, axes=([1], [0]))
            p_R = p_R / (N_R + 1e-10)
            p_R = np.where(valid, p_R, 0)
            H += -(N_L / N) * np.where(p_L > 0, p_L * np.log(p_L + 1e-15), 0)
            H += -(N_R / N) * np.where(p_R > 0, p_R * np.log(p_R + 1e-15), 0)

        H = np.where(valid, H, np.inf)
        best_idx = H.argmin(axis=0)
        dft_losses = H[best_idx, np.arange(P)]
        best_thresholds = thresholds[best_idx, np.arange(P)]

        n_classes = len(classes)
        dft_losses[degenerate] = np.log(n_classes) if n_classes > 1 else 0

        return dft_losses, best_thresholds

    def fit(self, X, y):
        X, y = check_X_y(X, y, accept_sparse=False, dtype=np.float64)
        if self.verbose:
            print(f"[DFT] Computing discriminant power for {X.shape[1]} features (B={self.n_bins})...")

        self.dft_losses_, self.thresholds_ = self._dft_fast(X, y)
        self.ranking_ = np.argsort(self.dft_losses_)

        if self.k == 'auto':
            self.k_ = self._find_elbow_k()
        else:
            self.k_ = min(int(self.k), X.shape[1])

        self.selected_features_ = self.ranking_[:self.k_]
        if self.verbose:
            print(f"[DFT] Selected {self.k_} features")
        return self

    def _find_elbow_k(self):
        power = 1.0 / (self.dft_losses_ + 1e-10)
        power_sorted = power[self.ranking_]
        cum_power = np.cumsum(power_sorted)
        cum_power_norm = cum_power / cum_power[-1]
        k = np.searchsorted(cum_power_norm, self.elbow_factor) + 1
        return max(1, min(k, len(self.dft_losses_)))

    def transform(self, X):
        X = check_array(X, accept_sparse=False, dtype=np.float64)
        return X[:, self.selected_features_]


# =============================================================================
# SECTION 3: STAGE 3 — SUBSPACE LEARNING MACHINE (SLM)
# =============================================================================

class SLMNode:
    def __init__(self, depth=0, node_id=0):
        self.depth = depth
        self.node_id = node_id
        self.is_leaf = False
        self.prediction = None
        self.projection = None
        self.threshold = None
        self.children = []
        self.subspace_indices = None
        self.class_distribution = None
        self.n_samples = 0

class SubspaceLearningMachine(BaseEstimator, ClassifierMixin):
    """Stage 3: Subspace Learning Machine (SLM) for classification."""
    def __init__(self, max_depth=5, min_samples_leaf=10,
                 purity_threshold=0.95, n_projections=50,
                 n_splits_per_node=1, max_features_per_node=None,
                 n_bins=16, random_state=42):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.purity_threshold = purity_threshold
        self.n_projections = n_projections
        self.n_splits_per_node = n_splits_per_node
        self.max_features_per_node = max_features_per_node
        self.n_bins = n_bins
        self.random_state = random_state

    def fit(self, X, y):
        self.rng = np.random.RandomState(self.random_state)
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}
        y_idx = np.array([self.class_to_idx_[c] for c in y])

        n_samples, n_features = X.shape

        if self.max_features_per_node is None or n_features <= self.max_features_per_node:
            subspace = np.arange(n_features)
        else:
            dft_scores = self._dft_scores(X, y_idx)
            subspace = np.argsort(dft_scores)[:self.max_features_per_node]

        self.node_counter = 0
        self.tree_ = self._build_tree(X, y_idx, np.ones(n_samples, dtype=bool),
                                       subspace, depth=0)
        return self

    def _dft_scores(self, X, y):
        N, P = X.shape
        f_min = X.min(axis=0)
        f_max = X.max(axis=0)
        degenerate = (f_max - f_min) < 1e-10

        b_vals = np.arange(1, self.n_bins).reshape(-1, 1)
        thresholds = f_min + (b_vals / self.n_bins) * (f_max - f_min)

        classes = np.unique(y)
        X_exp = X[np.newaxis, :, :]
        T_exp = thresholds[:, np.newaxis, :]

        left_mask = X_exp < T_exp
        right_mask = ~left_mask
        N_L = left_mask.sum(axis=1)
        N_R = right_mask.sum(axis=1)
        valid = (N_L > 0) & (N_R > 0)

        H = np.zeros((self.n_bins - 1, P))
        for c in classes:
            y_eq_c = (y == c).astype(np.float64)
            p_L = np.tensordot(left_mask.astype(np.float64), y_eq_c, axes=([1], [0]))
            p_L = p_L / (N_L + 1e-10)
            p_L = np.where(valid, p_L, 0)
            p_R = np.tensordot(right_mask.astype(np.float64), y_eq_c, axes=([1], [0]))
            p_R = p_R / (N_R + 1e-10)
            p_R = np.where(valid, p_R, 0)
            H += -(N_L / N) * np.where(p_L > 0, p_L * np.log(p_L + 1e-15), 0)
            H += -(N_R / N) * np.where(p_R > 0, p_R * np.log(p_R + 1e-15), 0)

        H = np.where(valid, H, np.inf)
        dft_losses = H.min(axis=0)
        dft_losses[degenerate] = np.log(len(classes)) if len(classes) > 1 else 0
        return dft_losses

    def _build_tree(self, X, y, mask, subspace, depth):
        node = SLMNode(depth=depth, node_id=self.node_counter)
        self.node_counter += 1
        node.subspace_indices = subspace

        y_node = y[mask]
        n_samples = len(y_node)
        node.n_samples = n_samples

        counts = np.bincount(y_node, minlength=self.n_classes_)
        node.class_distribution = counts / n_samples
        majority_class = counts.argmax()
        purity = counts[majority_class] / n_samples

        if (depth >= self.max_depth or
            n_samples < self.min_samples_leaf * 2 or
            purity >= self.purity_threshold or
            len(np.unique(y_node)) == 1):
            node.is_leaf = True
            node.prediction = self.classes_[majority_class]
            return node

        X_node = X[mask][:, subspace]
        projections, thresholds, losses = self._find_projections(X_node, y_node, subspace)

        if len(projections) == 0:
            node.is_leaf = True
            node.prediction = self.classes_[majority_class]
            return node

        q = min(self.n_splits_per_node, len(projections))
        best_idx = np.argsort(losses)[:q]

        node.projection = projections[best_idx[0]]
        node.threshold = thresholds[best_idx[0]]

        if q == 1:
            proj = projections[best_idx[0]]
            thresh = thresholds[best_idx[0]]
            projected = X_node @ proj
            left_mask_node = projected < thresh
            right_mask_node = ~left_mask_node

            if left_mask_node.sum() < self.min_samples_leaf or right_mask_node.sum() < self.min_samples_leaf:
                node.is_leaf = True
                node.prediction = self.classes_[majority_class]
                return node

            left_mask = mask.copy()
            left_mask[mask] = left_mask_node
            right_mask = mask.copy()
            right_mask[mask] = right_mask_node

            node.children.append(self._build_tree(X, y, left_mask, subspace, depth + 1))
            node.children.append(self._build_tree(X, y, right_mask, subspace, depth + 1))

        return node

    def _find_projections(self, X, y, subspace):
        n_samples, sub_dim = X.shape
        classes = np.unique(y)
        projections = []
        thresholds = []
        losses = []

        dft_scores = self._dft_scores(X, y)
        top_features = np.argsort(dft_scores)[:min(5, sub_dim)]

        for feat_idx in top_features:
            a = np.zeros(sub_dim)
            a[feat_idx] = 1.0
            proj_1d = X @ a
            loss, thresh = self._best_split_1d(proj_1d, y, classes)
            if loss < np.inf:
                projections.append(a)
                thresholds.append(thresh)
                losses.append(loss)

        for _ in range(self.n_projections):
            n_active = self.rng.randint(2, min(6, sub_dim + 1))
            active = self.rng.choice(sub_dim, size=n_active, replace=False)

            weights = 1.0 / (dft_scores[active] + 1e-10)
            weights = weights / weights.sum()

            a = np.zeros(sub_dim)
            a[active] = self.rng.randn(n_active) * weights
            a = a / (np.linalg.norm(a) + 1e-10)

            proj_1d = X @ a
            loss, thresh = self._best_split_1d(proj_1d, y, classes)
            if loss < np.inf:
                projections.append(a)
                thresholds.append(thresh)
                losses.append(loss)

        return projections, thresholds, losses

    def _best_split_1d(self, x_proj, y, classes):
        N = len(x_proj)
        f_min, f_max = x_proj.min(), x_proj.max()
        if f_max - f_min < 1e-10:
            return np.inf, f_min

        thresholds = f_min + np.arange(1, self.n_bins) / self.n_bins * (f_max - f_min)
        best_loss = np.inf
        best_thresh = thresholds[0]

        for t in thresholds:
            left_mask = x_proj < t
            right_mask = ~left_mask
            N_L, N_R = left_mask.sum(), right_mask.sum()
            if N_L == 0 or N_R == 0:
                continue

            H_L = sum(-(np.sum(y[left_mask] == c) / N_L) *
                      np.log(np.sum(y[left_mask] == c) / N_L + 1e-15)
                      for c in classes if np.sum(y[left_mask] == c) > 0)
            H_R = sum(-(np.sum(y[right_mask] == c) / N_R) *
                      np.log(np.sum(y[right_mask] == c) / N_R + 1e-15)
                      for c in classes if np.sum(y[right_mask] == c) > 0)

            H_t = (N_L / N) * H_L + (N_R / N) * H_R
            if H_t < best_loss:
                best_loss = H_t
                best_thresh = t

        return best_loss, best_thresh

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        return np.array([self._predict_single(x) for x in X])

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float64)
        return np.array([self._predict_proba_single(x) for x in X])

    def _predict_single(self, x):
        node = self.tree_
        while not node.is_leaf and len(node.children) > 0:
            sub_x = x[node.subspace_indices]
            if node.projection is not None and node.threshold is not None:
                proj_val = sub_x @ node.projection
                if proj_val < node.threshold:
                    node = node.children[0]
                else:
                    node = node.children[1]
            else:
                node = node.children[0]
        return node.prediction

    def _predict_proba_single(self, x):
        node = self.tree_
        while not node.is_leaf and len(node.children) > 0:
            sub_x = x[node.subspace_indices]
            if node.projection is not None and node.threshold is not None:
                proj_val = sub_x @ node.projection
                if proj_val < node.threshold:
                    node = node.children[0]
                else:
                    node = node.children[1]
            else:
                node = node.children[0]
        return node.class_distribution


class SLMForest(BaseEstimator, ClassifierMixin):
    """SLM Forest: Bootstrap aggregation of SLM trees."""
    def __init__(self, n_estimators=30, max_depth=5, min_samples_leaf=10,
                 n_projections=30, max_features='sqrt', n_bins=16,
                 random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.n_projections = n_projections
        self.max_features = max_features
        self.n_bins = n_bins
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.trees_ = []
        self.rng = np.random.RandomState(self.random_state)
        n_samples, n_features = X.shape

        max_feat = n_features
        if self.max_features == 'sqrt':
            max_feat = int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            max_feat = int(np.log2(n_features))
        elif isinstance(self.max_features, int):
            max_feat = self.max_features

        for i in range(self.n_estimators):
            idx = self.rng.choice(n_samples, size=n_samples, replace=True)
            X_boot, y_boot = X[idx], y[idx]

            tree = SubspaceLearningMachine(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                n_projections=self.n_projections,
                max_features_per_node=max_feat,
                n_bins=self.n_bins,
                random_state=self.rng.randint(100000)
            )
            tree.fit(X_boot, y_boot)
            self.trees_.append(tree)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        all_preds = np.array([tree.predict(X) for tree in self.trees_])
        predictions = []
        for i in range(X.shape[0]):
            votes = Counter(all_preds[:, i])
            predictions.append(votes.most_common(1)[0][0])
        return np.array(predictions)

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float64)
        all_probs = np.array([tree.predict_proba(X) for tree in self.trees_])
        return all_probs.mean(axis=0)


class SLMBoost(BaseEstimator, ClassifierMixin):
    """SLM Boost: Boosted ensemble of SLM trees."""
    def __init__(self, n_estimators=50, max_depth=3, min_samples_leaf=10,
                 n_projections=20, learning_rate=0.1, n_bins=16,
                 random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.n_projections = n_projections
        self.learning_rate = learning_rate
        self.n_bins = n_bins
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}
        y_idx = np.array([self.class_to_idx_[c] for c in y])

        n_samples = X.shape[0]
        sample_weights = np.ones(n_samples) / n_samples
        self.trees_ = []
        self.tree_weights_ = []
        self.rng = np.random.RandomState(self.random_state)

        for i in range(self.n_estimators):
            idx = self.rng.choice(n_samples, size=n_samples, replace=True, p=sample_weights)
            X_boot, y_boot = X[idx], y[idx]

            tree = SubspaceLearningMachine(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                n_projections=self.n_projections,
                n_bins=self.n_bins,
                random_state=self.rng.randint(100000)
            )
            tree.fit(X_boot, y_boot)

            y_pred = tree.predict(X)
            incorrect = (y_pred != y).astype(float)
            error = np.dot(sample_weights, incorrect)

            if error > 0.5:
                break

            tree_weight = self.learning_rate * np.log((1 - error) / (error + 1e-10))
            sample_weights *= np.exp(tree_weight * incorrect)
            sample_weights /= sample_weights.sum()

            self.trees_.append(tree)
            self.tree_weights_.append(tree_weight)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        all_probs = np.zeros((X.shape[0], self.n_classes_))
        for tree, weight in zip(self.trees_, self.tree_weights_):
            probs = tree.predict_proba(X)
            all_probs += weight * probs
        pred_idx = all_probs.argmax(axis=1)
        return self.classes_[pred_idx]


# =============================================================================
# SECTION 4: GREEN LEARNING PIPELINE CLASS
# =============================================================================

class GreenLearningPipeline:
    def __init__(self,
                 variance_threshold=0.95,
                 dft_bins=16,
                 dft_k='auto',
                 slm_max_depth=5,
                 slm_min_samples_leaf=10,
                 slm_n_projections=50,
                 ensemble='forest',
                 n_estimators=30,
                 verbose=True):
        self.variance_threshold = variance_threshold
        self.dft_bins = dft_bins
        self.dft_k = dft_k
        self.slm_max_depth = slm_max_depth
        self.slm_min_samples_leaf = slm_min_samples_leaf
        self.slm_n_projections = slm_n_projections
        self.ensemble = ensemble
        self.n_estimators = n_estimators
        self.verbose = verbose

        self.stage1 = None
        self.stage2 = None
        self.stage3 = None
        self.label_enc = None
        self.feature_names = None
        self.training_time = 0.0

    def fit(self, X, y, feature_names=None):
        t_start = time.time()
        self.feature_names = feature_names
        self.label_enc = LabelEncoder()
        y_enc = self.label_enc.fit_transform(y)

        if self.verbose:
            print("=" * 60)
            print("GREEN LEARNING PIPELINE")
            print("=" * 60)
            print(f"Classes: {self.label_enc.classes_}")
            print(f"Training samples: {X.shape[0]}, Features: {X.shape[1]}")

        # Stage 1
        if self.verbose:
            print("\n[STAGE 1] Unsupervised Subspace Approximation (PCA)...")
        self.stage1 = SubspaceApproximation(variance_threshold=self.variance_threshold)
        X_stage1 = self.stage1.fit_transform(X)
        if self.verbose:
            print(f"  Reduced {X.shape[1]} → {X_stage1.shape[1]} dims "
                  f"({self.stage1.explained_variance_:.2%} variance)")

        # Stage 2
        if self.verbose:
            print("\n[STAGE 2] Discriminant Feature Test (DFT)...")
        self.stage2 = DiscriminantFeatureTest(
            n_bins=self.dft_bins, k=self.dft_k, verbose=self.verbose
        )
        X_stage2 = self.stage2.fit_transform(X_stage1, y_enc)
        if self.verbose:
            print(f"  Selected {X_stage2.shape[1]} discriminant features")

        # Stage 3
        if self.verbose:
            print(f"\n[STAGE 3] Supervised Decision Learning ({self.ensemble})...")
        if self.ensemble == 'single':
            self.stage3 = SubspaceLearningMachine(
                max_depth=self.slm_max_depth, min_samples_leaf=self.slm_min_samples_leaf,
                n_projections=self.slm_n_projections, random_state=RANDOM_STATE
            )
        elif self.ensemble == 'forest':
            self.stage3 = SLMForest(
                n_estimators=self.n_estimators, max_depth=self.slm_max_depth,
                min_samples_leaf=self.slm_min_samples_leaf, n_projections=self.slm_n_projections,
                random_state=RANDOM_STATE
            )
        elif self.ensemble == 'boost':
            self.stage3 = SLMBoost(
                n_estimators=self.n_estimators, max_depth=self.slm_max_depth,
                min_samples_leaf=self.slm_min_samples_leaf, n_projections=self.slm_n_projections,
                random_state=RANDOM_STATE
            )

        self.stage3.fit(X_stage2, y_enc)
        self.training_time = time.time() - t_start

        if self.verbose:
            print(f"\n[Done] Total pipeline time: {self.training_time:.3f}s")
        return self

    def predict(self, X):
        X_stage1 = self.stage1.transform(X)
        X_stage2 = self.stage2.transform(X_stage1)
        y_pred_enc = self.stage3.predict(X_stage2)
        return self.label_enc.inverse_transform(y_pred_enc)

    def evaluate(self, X, y):
        y_pred = self.predict(X)
        acc = accuracy_score(y, y_pred)
        f1_macro = f1_score(y, y_pred, average='macro')
        f1_weighted = f1_score(y, y_pred, average='weighted')

        print("\n" + "=" * 60)
        print("EVALUATION RESULTS")
        print("=" * 60)
        print(f"Accuracy:       {acc:.4f}")
        print(f"F1 (macro):     {f1_macro:.4f}")
        print(f"F1 (weighted):  {f1_weighted:.4f}")
        print(f"\nClassification Report:")
        print(classification_report(y, y_pred, digits=4))
        return {'accuracy': acc, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted}


# =============================================================================
# SECTION 5: MAIN EXECUTION
# =============================================================================

def load_preprocessed_data("./content/iot23_train.csv", "./content/iot23_test.csv"):
    """Load previously preprocessed dataset into Memory"""
    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Training dataset not found at {train_path}. Run the preprocessor first.")

    print(f"[Load] Loading preprocessed training data: {train_path}")
    df_train = pd.read_csv(train_path)

    if test_path and os.path.exists(test_path):
        print(f"[Load] Loading preprocessed test data: {test_path}")
        df_test = pd.read_csv(test_path)
        return df_train, df_test

    print("[Load] No test dataset provided. Data will be dynamically split.")
    return df_train, None

def main():
    parser = argparse.ArgumentParser(description='Green Learning IoT-23 Classifier')
    parser.add_argument('--train_path', type=str, default='./iot23_processed/iot23_train.csv',
                        help='Path to the preprocessed training CSV')
    parser.add_argument('--test_path', type=str, default='./iot23_processed/iot23_test.csv',
                        help='Path to the preprocessed testing CSV')
    parser.add_argument('--ensemble', choices=['single', 'forest', 'boost'], default='forest',
                        help='SLM ensemble type')
    parser.add_argument('--max_depth', type=int, default=5)
    parser.add_argument('--n_estimators', type=int, default=30)
    parser.add_argument('--no_verbose', action='store_true')
    args = parser.parse_args()

    verbose = not args.no_verbose

    # 1. Load Data
    df_train, df_test = load_preprocessed_data(args.train_path, args.test_path)

    # 2. Automatically grab feature columns (all columns except 'label')
    feature_cols = [col for col in df_train.columns if col != 'label']

    # 3. Create Training and Testing arrays
    if df_test is not None:
        X_train = df_train[feature_cols].values
        y_train = df_train['label'].values
        X_test = df_test[feature_cols].values
        y_test = df_test['label'].values
    else:
        # Fallback to random splitting if test data file wasn't found
        X = df_train[feature_cols].values
        y = df_train['label'].values
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )

    if verbose:
        print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")
        print(f"Detected {len(feature_cols)} features.")

    # 4. Build and train Green Learning Pipeline
    gl = GreenLearningPipeline(
        variance_threshold=0.95,
        dft_bins=16,
        dft_k='auto',
        slm_max_depth=args.max_depth,
        slm_min_samples_leaf=10,
        slm_n_projections=50,
        ensemble=args.ensemble,
        n_estimators=args.n_estimators,
        verbose=verbose
    )
    gl.fit(X_train, y_train, feature_names=feature_cols)

    # 5. Evaluate GL Model
    results = gl.evaluate(X_test, y_test)

    # 6. Baseline comparisons
    if verbose:
        print("\n" + "=" * 60)
        print("BASELINE COMPARISONS")
        print("=" * 60)

        dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=RANDOM_STATE)
        dt.fit(X_train, y_train)
        dt_acc = accuracy_score(y_test, dt.predict(X_test))
        print(f"Decision Tree:     {dt_acc:.4f}")

        rf = RandomForestClassifier(n_estimators=30, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1)
        rf.fit(X_train, y_train)
        rf_acc = accuracy_score(y_test, rf.predict(X_test))
        print(f"Random Forest:     {rf_acc:.4f}")

        gb = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=RANDOM_STATE)
        gb.fit(X_train, y_train)
        gb_acc = accuracy_score(y_test, gb.predict(X_test))
        print(f"Gradient Boosting: {gb_acc:.4f}")

        print(f"\nGreen Learning ({args.ensemble}): {results['accuracy']:.4f}")

if __name__ == "__main__":
    main()

SyntaxError: invalid syntax (3069766813.py, line 696)

### Addressing Class Imbalance

As observed in the classification report, classes like 'FileDownload' and 'Torii' have 0.0000 F1-scores. This is a classic sign of a **severely imbalanced dataset**, where these classes have very few samples, making it difficult for the model to learn to predict them.

To mitigate this, we can use techniques like **oversampling** the minority classes. One popular method is **Synthetic Minority Over-sampling Technique (SMOTE)**, which generates synthetic samples for the minority class. Before applying SMOTE, let's explicitly load the data and examine the original class distribution in the training set.

In [ ]:
# Extracting data loading and splitting logic from main()

# Simulate argparse arguments for interactive use
class Args:
    def __init__(self):
        self.train_path = './iot23_train.csv' # Corrected path
        self.test_path = './iot23_test.csv'   # Corrected path
        self.ensemble = 'xgboost'
        self.max_depth = 5
        self.n_estimators = 30
        self.no_verbose = False

args = Args()

# 1. Load Data
df_train, df_test = load_preprocessed_data(args.train_path, args.test_path)

# 2. Automatically grab feature columns (all columns except 'label')
feature_cols = [col for col in df_train.columns if col != 'label']

[Load] Loading preprocessed training data: ./iot23_train.csv
[Load] Loading preprocessed test data: ./iot23_test.csv


In [ ]:
# 3. Create Training and Testing arrays
if df_test is not None:
    X_train = df_train[feature_cols].values
    y_train = df_train['label'].values
    X_test = df_test[feature_cols].values
    y_test = df_test['label'].values
else:
    # Fallback to random splitting if test data file wasn't found
    X = df_train[feature_cols].values
    y = df_train['label'].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Detected {len(feature_cols)} features.")

Train shape: (233988, 36), Test shape: (77996, 36)
Detected 36 features.


#### Original Class Distribution in Training Data

In [ ]:
from collections import Counter

print("Original training class distribution:")
original_counts = Counter(y_train)
for cls, count in original_counts.most_common():
    print(f"  {cls}: {count}")

Original training class distribution:
  Benign: 91536
  PortScan: 52683
  DDoS: 37501
  Okiru: 22500
  C&C: 18059
  Attack: 11638
  Torii: 45
  FileDownload: 26


#### Applying SMOTE to Balance the Training Data

We will now use SMOTE to oversample the minority classes in the training set. This will create synthetic samples for the underrepresented classes, helping the model learn their patterns better.

In [ ]:
from imblearn.over_sampling import SMOTE

print("Applying SMOTE...")
smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("New training class distribution after SMOTE:")
resampled_counts = Counter(y_train_resampled)
for cls, count in resampled_counts.most_common():
    print(f"  {cls}: {count}")

print(f"Resampled training shape: {X_train_resampled.shape}")

Applying SMOTE...
New training class distribution after SMOTE:
  Okiru: 91536
  Benign: 91536
  DDoS: 91536
  C&C: 91536
  PortScan: 91536
  Attack: 91536
  Torii: 91536
  FileDownload: 91536
Resampled training shape: (732288, 36)


In [ ]:
import os
import sys
import argparse
import time
import warnings
from collections import Counter
from typing import Tuple, List, Optional

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.feature_selection import SelectorMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, accuracy_score, f1_score)
from sklearn.utils.validation import check_X_y, check_array

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# IoT-23 Unified Classes (Matches the Preprocessor Output)
IOT23_CLASSES = [
    'Benign',
    'Attack',
    'C&C',
    'DDoS',
    'FileDownload',
    'Okiru',
    'PortScan',
    'Torii'
]


# =============================================================================
# SECTION 1: STAGE 1 — UNSUPERVISED REPRESENTATION LEARNING (PCA)
# =============================================================================

class SubspaceApproximation(BaseEstimator, TransformerMixin):
    """
    Stage 1: Unsupervised Subspace Approximation via PCA.
    Reduces spectral dimension while retaining specified variance.
    """
    def __init__(self, variance_threshold: float = 0.95):
        self.variance_threshold = variance_threshold
        self.scaler = StandardScaler()
        self.pca = None
        self.n_components_ = None
        self.explained_variance_ = None

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=np.float64)
        X_scaled = self.scaler.fit_transform(X)
        self.pca = PCA(n_components=self.variance_threshold, random_state=RANDOM_STATE)
        self.pca.fit(X_scaled)
        self.n_components_ = self.pca.n_components_
        self.explained_variance_ = self.pca.explained_variance_ratio_.sum()
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        X_scaled = self.scaler.transform(X)
        return self.pca.transform(X_scaled)

    def fit_transform(self, X, y=None):
        self.fit(X)
        return self.transform(X)


# =============================================================================
# SECTION 2: STAGE 2 — DISCRIMINANT FEATURE TEST (DFT)
# =============================================================================

class DiscriminantFeatureTest(BaseEstimator, SelectorMixin, TransformerMixin):
    """
    Stage 2: Discriminant Feature Test (DFT) for supervised feature selection.
    Evaluates each 1-D feature by partitioning its range and computing the
    minimum weighted entropy across candidate thresholds. Lower DFT loss
    = higher discriminant power.
    """
    def __init__(self, n_bins: int = 16, k: str = 'auto',
                 elbow_factor: float = 0.95, verbose: bool = False):
        self.n_bins = n_bins
        self.k = k
        self.elbow_factor = elbow_factor
        self.verbose = verbose

    def _dft_fast(self, X, y):
        N, P = X.shape
        f_min = X.min(axis=0)
        f_max = X.max(axis=0)
        degenerate = (f_max - f_min) < 1e-10

        b_vals = np.arange(1, self.n_bins).reshape(-1, 1)
        thresholds = f_min + (b_vals / self.n_bins) * (f_max - f_min)

        classes = np.unique(y)
        X_exp = X[np.newaxis, :, :]
        T_exp = thresholds[:, np.newaxis, :]

        left_mask = X_exp < T_exp
        right_mask = ~left_mask
        N_L = left_mask.sum(axis=1)
        N_R = right_mask.sum(axis=1)
        valid = (N_L > 0) & (N_R > 0)

        H = np.zeros((self.n_bins - 1, P))

        for c in classes:
            y_eq_c = (y == c).astype(np.float64)
            p_L = np.tensordot(left_mask.astype(np.float64), y_eq_c, axes=([1], [0]))
            p_L = p_L / (N_L + 1e-10)
            p_L = np.where(valid, p_L, 0)
            p_R = np.tensordot(right_mask.astype(np.float64), y_eq_c, axes=([1], [0]))
            p_R = p_R / (N_R + 1e-10)
            p_R = np.where(valid, p_R, 0)
            H += -(N_L / N) * np.where(p_L > 0, p_L * np.log(p_L + 1e-15), 0)
            H += -(N_R / N) * np.where(p_R > 0, p_R * np.log(p_R + 1e-15), 0)

        H = np.where(valid, H, np.inf)
        best_idx = H.argmin(axis=0)
        dft_losses = H[best_idx, np.arange(P)]
        best_thresholds = thresholds[best_idx, np.arange(P)]

        n_classes = len(classes)
        dft_losses[degenerate] = np.log(n_classes) if n_classes > 1 else 0

        return dft_losses, best_thresholds

    def fit(self, X, y):
        X, y = check_X_y(X, y, accept_sparse=False, dtype=np.float64)
        if self.verbose:
            print(f"[DFT] Computing discriminant power for {X.shape[1]} features (B={self.n_bins})...")

        self.dft_losses_, self.thresholds_ = self._dft_fast(X, y)
        self.ranking_ = np.argsort(self.dft_losses_)

        if self.k == 'auto':
            self.k_ = self._find_elbow_k()
        else:
            self.k_ = min(int(self.k), X.shape[1])

        self.selected_features_ = self.ranking_[:self.k_]
        if self.verbose:
            print(f"[DFT] Selected {self.k_} features")
        return self

    def _find_elbow_k(self):
        power = 1.0 / (self.dft_losses_ + 1e-10)
        power_sorted = power[self.ranking_]
        cum_power = np.cumsum(power_sorted)
        cum_power_norm = cum_power / cum_power[-1]
        k = np.searchsorted(cum_power_norm, self.elbow_factor) + 1
        return max(1, min(k, len(self.dft_losses_)))

    def transform(self, X):
        X = check_array(X, accept_sparse=False, dtype=np.float64)
        return X[:, self.selected_features_]

    def _get_support_mask(self):
        # Required by SelectorMixin
        mask = np.zeros(self.ranking_.shape, dtype=bool)
        mask[self.selected_features_] = True
        return mask


# =============================================================================
# SECTION 3: STAGE 3 — SUBSPACE LEARNING MACHINE (SLM)
# =============================================================================

class SLMNode:
    def __init__(self, depth=0, node_id=0):
        self.depth = depth
        self.node_id = node_id
        self.is_leaf = False
        self.prediction = None
        self.projection = None
        self.threshold = None
        self.children = []
        self.subspace_indices = None
        self.class_distribution = None
        self.n_samples = 0

class SubspaceLearningMachine(BaseEstimator, ClassifierMixin):
    """Stage 3: Subspace Learning Machine (SLM) for classification."""
    def __init__(self, max_depth=5, min_samples_leaf=10,
                 purity_threshold=0.95, n_projections=50,
                 n_splits_per_node=1, max_features_per_node=None,
                 n_bins=16, random_state=42):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.purity_threshold = purity_threshold
        self.n_projections = n_projections
        self.n_splits_per_node = n_splits_per_node
        self.max_features_per_node = max_features_per_node
        self.n_bins = n_bins
        self.random_state = random_state

    def fit(self, X, y):
        self.rng = np.random.RandomState(self.random_state)
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}
        y_idx = np.array([self.class_to_idx_[c] for c in y])

        n_samples, n_features = X.shape

        if self.max_features_per_node is None or n_features <= self.max_features_per_node:
            subspace = np.arange(n_features)
        else:
            dft_scores = self._dft_scores(X, y_idx)
            subspace = np.argsort(dft_scores)[:self.max_features_per_node]

        self.node_counter = 0
        self.tree_ = self._build_tree(X, y_idx, np.ones(n_samples, dtype=bool),
                                       subspace, depth=0)
        return self

    def _dft_scores(self, X, y):
        N, P = X.shape
        f_min = X.min(axis=0)
        f_max = X.max(axis=0)
        degenerate = (f_max - f_min) < 1e-10

        b_vals = np.arange(1, self.n_bins).reshape(-1, 1)
        thresholds = f_min + (b_vals / self.n_bins) * (f_max - f_min)

        classes = np.unique(y)
        X_exp = X[np.newaxis, :, :]
        T_exp = thresholds[:, np.newaxis, :]

        left_mask = X_exp < T_exp
        right_mask = ~left_mask
        N_L = left_mask.sum(axis=1)
        N_R = right_mask.sum(axis=1)
        valid = (N_L > 0) & (N_R > 0)

        H = np.zeros((self.n_bins - 1, P))
        for c in classes:
            y_eq_c = (y == c).astype(np.float64)
            p_L = np.tensordot(left_mask.astype(np.float64), y_eq_c, axes=([1], [0]))
            p_L = p_L / (N_L + 1e-10)
            p_L = np.where(valid, p_L, 0)
            p_R = np.tensordot(right_mask.astype(np.float64), y_eq_c, axes=([1], [0]))
            p_R = p_R / (N_R + 1e-10)
            p_R = np.where(valid, p_R, 0)
            H += -(N_L / N) * np.where(p_L > 0, p_L * np.log(p_L + 1e-15), 0)
            H += -(N_R / N) * np.where(p_R > 0, p_R * np.log(p_R + 1e-15), 0)

        H = np.where(valid, H, np.inf)
        dft_losses = H.min(axis=0)
        dft_losses[degenerate] = np.log(len(classes)) if len(classes) > 1 else 0
        return dft_losses

    def _build_tree(self, X, y, mask, subspace, depth):
        node = SLMNode(depth=depth, node_id=self.node_counter)
        self.node_counter += 1
        node.subspace_indices = subspace

        y_node = y[mask]
        n_samples = len(y_node)
        node.n_samples = n_samples

        counts = np.bincount(y_node, minlength=self.n_classes_)
        node.class_distribution = counts / n_samples
        majority_class = counts.argmax()
        purity = counts[majority_class] / n_samples

        if (depth >= self.max_depth or
            n_samples < self.min_samples_leaf * 2 or
            purity >= self.purity_threshold or
            len(np.unique(y_node)) == 1):
            node.is_leaf = True
            node.prediction = self.classes_[majority_class]
            return node

        X_node = X[mask][:, subspace]
        projections, thresholds, losses = self._find_projections(X_node, y_node, subspace)

        if len(projections) == 0:
            node.is_leaf = True
            node.prediction = self.classes_[majority_class]
            return node

        q = min(self.n_splits_per_node, len(projections))
        best_idx = np.argsort(losses)[:q]

        node.projection = projections[best_idx[0]]
        node.threshold = thresholds[best_idx[0]]

        if q == 1:
            proj = projections[best_idx[0]]
            thresh = thresholds[best_idx[0]]
            projected = X_node @ proj
            left_mask_node = projected < thresh
            right_mask_node = ~left_mask_node

            if left_mask_node.sum() < self.min_samples_leaf or right_mask_node.sum() < self.min_samples_leaf:
                node.is_leaf = True
                node.prediction = self.classes_[majority_class]
                return node

            left_mask = mask.copy()
            left_mask[mask] = left_mask_node
            right_mask = mask.copy()
            right_mask[mask] = right_mask_node

            node.children.append(self._build_tree(X, y, left_mask, subspace, depth + 1))
            node.children.append(self._build_tree(X, y, right_mask, subspace, depth + 1))

        return node

    def _find_projections(self, X, y, subspace):
        n_samples, sub_dim = X.shape
        classes = np.unique(y)
        projections = []
        thresholds = []
        losses = []

        dft_scores = self._dft_scores(X, y)
        top_features = np.argsort(dft_scores)[:min(5, sub_dim)]

        for feat_idx in top_features:
            a = np.zeros(sub_dim)
            a[feat_idx] = 1.0
            proj_1d = X @ a
            loss, thresh = self._best_split_1d(proj_1d, y, classes)
            if loss < np.inf:
                projections.append(a)
                thresholds.append(thresh)
                losses.append(loss)

        for _ in range(self.n_projections):
            n_active = self.rng.randint(2, min(6, sub_dim + 1))
            active = self.rng.choice(sub_dim, size=n_active, replace=False)

            weights = 1.0 / (dft_scores[active] + 1e-10)
            weights = weights / weights.sum()

            a = np.zeros(sub_dim)
            a[active] = self.rng.randn(n_active) * weights
            a = a / (np.linalg.norm(a) + 1e-10)

            proj_1d = X @ a
            loss, thresh = self._best_split_1d(proj_1d, y, classes)
            if loss < np.inf:
                projections.append(a)
                thresholds.append(thresh)
                losses.append(loss)

        return projections, thresholds, losses

    def _best_split_1d(self, x_proj, y, classes):
        N = len(x_proj)
        f_min, f_max = x_proj.min(), x_proj.max()
        if f_max - f_min < 1e-10:
            return np.inf, f_min

        thresholds = f_min + np.arange(1, self.n_bins) / self.n_bins * (f_max - f_min)
        best_loss = np.inf
        best_thresh = thresholds[0]

        for t in thresholds:
            left_mask = x_proj < t
            right_mask = ~left_mask
            N_L, N_R = left_mask.sum(), right_mask.sum()
            if N_L == 0 or N_R == 0:
                continue

            H_L = sum(-(np.sum(y[left_mask] == c) / N_L) *
                      np.log(np.sum(y[left_mask] == c) / N_L + 1e-15)
                      for c in classes if np.sum(y[left_mask] == c) > 0)
            H_R = sum(-(np.sum(y[right_mask] == c) / N_R) *
                      np.log(np.sum(y[right_mask] == c) / N_R + 1e-15)
                      for c in classes if np.sum(y[right_mask] == c) > 0)

            H_t = (N_L / N) * H_L + (N_R / N) * H_R
            if H_t < best_loss:
                best_loss = H_t
                best_thresh = t

        return best_loss, best_thresh

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        return np.array([self._predict_single(x) for x in X])

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float64)
        return np.array([self._predict_proba_single(x) for x in X])

    def _predict_single(self, x):
        node = self.tree_
        while not node.is_leaf and len(node.children) > 0:
            sub_x = x[node.subspace_indices]
            if node.projection is not None and node.threshold is not None:
                proj_val = sub_x @ node.projection
                if proj_val < node.threshold:
                    node = node.children[0]
                else:
                    node = node.children[1]
            else:
                node = node.children[0]
        return node.prediction

    def _predict_proba_single(self, x):
        node = self.tree_
        while not node.is_leaf and len(node.children) > 0:
            sub_x = x[node.subspace_indices]
            if node.projection is not None and node.threshold is not None:
                proj_val = sub_x @ node.projection
                if proj_val < node.threshold:
                    node = node.children[0]
                else:
                    node = node.children[1]
            else:
                node = node.children[0]
        return node.class_distribution


class SLMForest(BaseEstimator, ClassifierMixin):
    """SLM Forest: Bootstrap aggregation of SLM trees."""
    def __init__(self, n_estimators=30, max_depth=5, min_samples_leaf=10,
                 n_projections=30, max_features='sqrt', n_bins=16,
                 random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.n_projections = n_projections
        self.max_features = max_features
        self.n_bins = n_bins
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.trees_ = []
        self.rng = np.random.RandomState(self.random_state)
        n_samples, n_features = X.shape

        max_feat = n_features
        if self.max_features == 'sqrt':
            max_feat = int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            max_feat = int(np.log2(n_features))
        elif isinstance(self.max_features, int):
            max_feat = self.max_features

        for i in range(self.n_estimators):
            idx = self.rng.choice(n_samples, size=n_samples, replace=True)
            X_boot, y_boot = X[idx], y[idx]

            tree = SubspaceLearningMachine(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                n_projections=self.n_projections,
                max_features_per_node=max_feat,
                n_bins=self.n_bins,
                random_state=self.rng.randint(100000)
            )
            tree.fit(X_boot, y_boot)
            self.trees_.append(tree)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        all_preds = np.array([tree.predict(X) for tree in self.trees_])
        predictions = []
        for i in range(X.shape[0]):
            votes = Counter(all_preds[:, i])
            predictions.append(votes.most_common(1)[0][0])
        return np.array(predictions)

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float64)
        all_probs = np.array([tree.predict_proba(X) for tree in self.trees_])
        return all_probs.mean(axis=0)


class SLMBoost(BaseEstimator, ClassifierMixin):
    """SLM Boost: Boosted ensemble of SLM trees."""
    def __init__(self, n_estimators=50, max_depth=3, min_samples_leaf=10,
                 n_projections=20, learning_rate=0.1, n_bins=16,
                 random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.n_projections = n_projections
        self.learning_rate = learning_rate
        self.n_bins = n_bins
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}
        y_idx = np.array([self.class_to_idx_[c] for c in y])

        n_samples = X.shape[0]
        sample_weights = np.ones(n_samples) / n_samples
        self.trees_ = []
        self.tree_weights_ = []
        self.rng = np.random.RandomState(self.random_state)

        for i in range(self.n_estimators):
            idx = self.rng.choice(n_samples, size=n_samples, replace=True, p=sample_weights)
            X_boot, y_boot = X[idx], y[idx]

            tree = SubspaceLearningMachine(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                n_projections=self.n_projections,
                n_bins=self.n_bins,
                random_state=self.rng.randint(100000)
            )
            tree.fit(X_boot, y_boot)

            y_pred = tree.predict(X)
            incorrect = (y_pred != y).astype(float)
            error = np.dot(sample_weights, incorrect)

            if error > 0.5:
                break

            tree_weight = self.learning_rate * np.log((1 - error) / (error + 1e-10))
            sample_weights *= np.exp(tree_weight * incorrect)
            sample_weights /= sample_weights.sum()

            self.trees_.append(tree)
            self.tree_weights_.append(tree_weight)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        all_probs = np.zeros((X.shape[0], self.n_classes_))
        for tree, weight in zip(self.trees_, self.tree_weights_):
            probs = tree.predict_proba(X)
            all_probs += weight * probs
        pred_idx = all_probs.argmax(axis=1)
        return self.classes_[pred_idx]


# =============================================================================
# SECTION 4: GREEN LEARNING PIPELINE CLASS
# =============================================================================

class GreenLearningPipeline:
    def __init__(self,
                 variance_threshold=0.95,
                 dft_bins=16,
                 dft_k='auto',
                 slm_max_depth=5,
                 slm_min_samples_leaf=10,
                 slm_n_projections=50,
                 ensemble='forest',
                 n_estimators=30,
                 verbose=True):
        self.variance_threshold = variance_threshold
        self.dft_bins = dft_bins
        self.dft_k = dft_k
        self.slm_max_depth = slm_max_depth
        self.slm_min_samples_leaf = slm_min_samples_leaf
        self.slm_n_projections = slm_n_projections
        self.ensemble = ensemble
        self.n_estimators = n_estimators
        self.verbose = verbose

        self.stage1 = None
        self.stage2 = None
        self.stage3 = None
        self.label_enc = None
        self.feature_names = None
        self.training_time = 0.0

    def fit(self, X, y, feature_names=None):
        t_start = time.time()
        self.feature_names = feature_names
        self.label_enc = LabelEncoder()
        y_enc = self.label_enc.fit_transform(y)

        if self.verbose:
            print("=" * 60)
            print("GREEN LEARNING PIPELINE")
            print("=" * 60)
            print(f"Classes: {self.label_enc.classes_}")
            print(f"Training samples: {X.shape[0]}, Features: {X.shape[1]}")

        # Stage 1
        if self.verbose:
            print("\n[STAGE 1] Unsupervised Subspace Approximation (PCA)...")
        self.stage1 = SubspaceApproximation(variance_threshold=self.variance_threshold)
        X_stage1 = self.stage1.fit_transform(X)
        if self.verbose:
            print(f"  Reduced {X.shape[1]} → {X_stage1.shape[1]} dims "
                  f"({self.stage1.explained_variance_:.2%} variance)")

        # Stage 2
        if self.verbose:
            print("\n[STAGE 2] Discriminant Feature Test (DFT)...")
        self.stage2 = DiscriminantFeatureTest(
            n_bins=self.dft_bins, k=self.dft_k, verbose=self.verbose
        )
        X_stage2 = self.stage2.fit_transform(X_stage1, y_enc)
        if self.verbose:
            print(f"  Selected {X_stage2.shape[1]} discriminant features")

        # Stage 3
        if self.verbose:
            print(f"\n[STAGE 3] Supervised Decision Learning ({self.ensemble})...")
        if self.ensemble == 'single':
            self.stage3 = SubspaceLearningMachine(
                max_depth=self.slm_max_depth, min_samples_leaf=self.slm_min_samples_leaf,
                n_projections=self.slm_n_projections, random_state=RANDOM_STATE
            )
        elif self.ensemble == 'forest':
            self.stage3 = SLMForest(
                n_estimators=self.n_estimators, max_depth=self.slm_max_depth,
                min_samples_leaf=self.slm_min_samples_leaf, n_projections=self.slm_n_projections,
                random_state=RANDOM_STATE
            )
        elif self.ensemble == 'boost':
            self.stage3 = SLMBoost(
                n_estimators=self.n_estimators, max_depth=self.slm_max_depth,
                min_samples_leaf=self.slm_min_samples_leaf, n_projections=self.slm_n_projections,
                random_state=RANDOM_STATE
            )

        self.stage3.fit(X_stage2, y_enc)
        self.training_time = time.time() - t_start

        if self.verbose:
            print(f"\n[Done] Total pipeline time: {self.training_time:.3f}s")
        return self

    def predict(self, X):
        X_stage1 = self.stage1.transform(X)
        X_stage2 = self.stage2.transform(X_stage1)
        y_pred_enc = self.stage3.predict(X_stage2)
        return self.label_enc.inverse_transform(y_pred_enc)

    def evaluate(self, X, y):
        y_pred = self.predict(X)
        acc = accuracy_score(y, y_pred)
        f1_macro = f1_score(y, y_pred, average='macro')
        f1_weighted = f1_score(y, y_pred, average='weighted')

        print("\n" + "=" * 60)
        print("EVALUATION RESULTS")
        print("=" * 60)
        print(f"Accuracy:       {acc:.4f}")
        print(f"F1 (macro):     {f1_macro:.4f}")
        print(f"F1 (weighted):  {f1_weighted:.4f}")
        print(f"\nClassification Report:")
        print(classification_report(y, y_pred, digits=4))
        return {'accuracy': acc, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted}


# =============================================================================
# SECTION 5: MAIN EXECUTION
# =============================================================================

def load_preprocessed_data(train_path: str = "./content/iot23_train.csv", test_path: str = "./content/iot23_test.csv"):
    """Load previously preprocessed dataset into Memory"""
    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Training dataset not found at {train_path}. Run the preprocessor first.")

    print(f"[Load] Loading preprocessed training data: {train_path}")
    df_train = pd.read_csv(train_path)

    if test_path and os.path.exists(test_path):
        print(f"[Load] Loading preprocessed test data: {test_path}")
        df_test = pd.read_csv(test_path)
        return df_train, df_test

    print("[Load] No test dataset provided. Data will be dynamically split.")
    return df_train, None

def main():
    parser = argparse.ArgumentParser(description='Green Learning IoT-23 Classifier')
    parser.add_argument('--train_path', type=str, default='./iot23_train.csv',
                        help='Path to the preprocessed training CSV')
    parser.add_argument('--test_path', type=str, default='./iot23_test.csv',
                        help='Path to the preprocessed testing CSV')
    parser.add_argument('--ensemble', choices=['single', 'forest', 'boost'], default='forest',
                        help='SLM ensemble type')
    parser.add_argument('--max_depth', type=int, default=5)
    parser.add_argument('--n_estimators', type=int, default=30)
    parser.add_argument('--no_verbose', action='store_true')
    args = parser.parse_args([]) # Pass an empty list to ignore kernel arguments

    verbose = not args.no_verbose

    # 1. Load Data
    df_train, df_test = load_preprocessed_data(args.train_path, args.test_path)

    # 2. Automatically grab feature columns (all columns except 'label')
    feature_cols = [col for col in df_train.columns if col != 'label']

    # 3. Create Training and Testing arrays
    if df_test is not None:
        X_train = df_train[feature_cols].values
        y_train = df_train['label'].values
        X_test = df_test[feature_cols].values
        y_test = df_test['label'].values
    else:
        # Fallback to random splitting if test data file wasn't found
        X = df_train[feature_cols].values
        y = df_train['label'].values
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )

    if verbose:
        print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")
        print(f"Detected {len(feature_cols)} features.")

    # 4. Build and train Green Learning Pipeline
    gl = GreenLearningPipeline(
        variance_threshold=0.95,
        dft_bins=16,
        dft_k='auto',
        slm_max_depth=args.max_depth,
        slm_min_samples_leaf=10,
        slm_n_projections=50,
        ensemble=args.ensemble,
        n_estimators=args.n_estimators,
        verbose=verbose
    )
    gl.fit(X_train, y_train, feature_names=feature_cols)

    # 5. Evaluate GL Model
    results = gl.evaluate(X_test, y_test)

    # 6. Baseline comparisons
    if verbose:
        print("\n" + "=" * 60)
        print("BASELINE COMPARISONS")
        print("=" * 60)

        dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=RANDOM_STATE)
        dt.fit(X_train, y_train)
        dt_acc = accuracy_score(y_test, dt.predict(X_test))
        print(f"Decision Tree:     {dt_acc:.4f}")

        rf = RandomForestClassifier(n_estimators=30, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1)
        rf.fit(X_train, y_train)
        rf_acc = accuracy_score(y_test, rf.predict(X_test))
        print(f"Random Forest:     {rf_acc:.4f}")

        gb = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=RANDOM_STATE)
        gb.fit(X_train, y_train)
        gb_acc = accuracy_score(y_test, gb.predict(X_test))
        print(f"Gradient Boosting: {gb_acc:.4f}")

        print(f"\nGreen Learning ({args.ensemble}): {results['accuracy']:.4f}")

if __name__ == "__main__":
    main()

[Load] Loading preprocessed training data: ./iot23_train.csv
[Load] Loading preprocessed test data: ./iot23_test.csv

Train shape: (233988, 36), Test shape: (77996, 36)
Detected 36 features.
GREEN LEARNING PIPELINE
Classes: ['Attack' 'Benign' 'C&C' 'DDoS' 'FileDownload' 'Okiru' 'PortScan' 'Torii']
Training samples: 233988, Features: 36

[STAGE 1] Unsupervised Subspace Approximation (PCA)...
  Reduced 36 → 16 dims (95.57% variance)

[STAGE 2] Discriminant Feature Test (DFT)...
[DFT] Computing discriminant power for 16 features (B=16)...
[DFT] Selected 16 features
  Selected 16 discriminant features

[STAGE 3] Supervised Decision Learning (forest)...

[Done] Total pipeline time: 4338.351s

EVALUATION RESULTS
Accuracy:       0.8864
F1 (macro):     0.6596
F1 (weighted):  0.8855

Classification Report:
              precision    recall  f1-score   support

      Attack     0.9850    0.6448    0.7794      3880
      Benign     0.8613    0.9103    0.8851     30512
         C&C     0.8014    0